In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import glob, os
import glob
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import GroupShuffleSplit




In [ ]:
def calculate_price_spread(df):
    """
    For test data: calculate price spread for every row.
    Computes price_spread as the difference between the maximum and minimum of all price columns,
    then adds spread values as 'price_spread' column to the dataframe.
    """    
    # calculate price spread for all rows
    df=df.copy()
    df['price_spread'] = df[['bid_price1', 'bid_price2', 'ask_price1', 'ask_price2']].max(axis=1) - df[['bid_price1', 'bid_price2', 'ask_price1', 'ask_price2']].min(axis=1)
    return df

def calculate_depth_imbalance(df):
    """
    Depth imbalance measures the relative asymmetry between bid and ask liquidity.
    Ranges from -1 (all ask-side) to +1 (all bid-side).
    Uses total bid and ask depth across both levels.
    """
    df = df.copy()
    total_bid = df['bid_size1'] + df['bid_size2']
    total_ask = df['ask_size1'] + df['ask_size2']
    df['depth_imbalance'] = (total_bid - total_ask) / (total_bid + total_ask)
    return df

def calculate_bid_ask_spread(df):
    """
    For test data: calculate bid-ask spread for every row.
    Computes best_bid_price and best_ask_price from raw price columns,
    then adds spread values as 'bid_ask_spread' column to the dataframe.
    """
    df = df.copy()
    
    # compute best bid/ask prices from raw columns for all rows
    df['best_bid_price'] = np.maximum(df['bid_price1'], df['bid_price2'])
    df['best_ask_price'] = np.minimum(df['ask_price1'], df['ask_price2'])
    
    # calculate bid-ask spread for all rows
    df['bid_ask_spread'] = (df['best_ask_price']/df['best_bid_price']) - 1
    
    # drop intermediate columns if not needed later
    df.drop(columns=['best_bid_price', 'best_ask_price'], inplace=True)   
    return df

def calculate_volume(df):
    """
    For test data: calculate total volume for every row.
    Computes best_bid_size and best_ask_size from raw size columns,
    then adds total volume values as 'total_volume' column to the dataframe.
    """
    df=df.copy()
    df['total_volume'] = df[['bid_size1', 'bid_size2', 'ask_size1', 'ask_size2']].sum(axis=1)
    
    # drop intermediate columns if not needed later
    return df


def add_wap_at_zero(df):
    df=df.copy()
    df['wap'] = (df['bid_price1'] * df['ask_size1'] + 
                        df['ask_price1'] * df['bid_size1']) / (
                        df['bid_size1'] + df['ask_size1'])
    return df



In [ ]:
out_dir = "individual_book_train_denorm"
os.makedirs(out_dir, exist_ok=True)


def fill_missing_seconds_in_bucket(df):
    """Ensure every time_id has rows for all 0..599 seconds_in_bucket, forward-filling values."""
    # Keep original order semantics per time_id and fill gaps with prior available row.
    df = df.sort_values(['time_id', 'seconds_in_bucket'])
    filled = (
        df.groupby('time_id', group_keys=False)
          .apply(lambda g: g.set_index('seconds_in_bucket')
                         .reindex(range(600))
                         .ffill()
                         .bfill()
                         .assign(time_id=g['time_id'].iloc[0]))
          .reset_index()
    )
    return filled


def process_file(path, out_dir):
    fname = os.path.basename(path)
    out_path = os.path.join(out_dir, fname)
    try:
        
        df = pd.read_csv(path)
        df = add_wap_at_zero(df)
        df = calculate_bid_ask_spread(df)
        df = calculate_volume(df)
        df = calculate_price_spread(df)
        df = calculate_depth_imbalance(df)
        df = df[['time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume', 'price_spread', 'depth_imbalance']]
        df = fill_missing_seconds_in_bucket(df)
        df.to_csv(out_path, index=False)
        return (fname, 'ok')
    except Exception as e:
        return (fname, f'error: {e}')

files = glob.glob(r"individual_book_train\stock_*.csv")
print(f"Found {len(files)} files to process")

# Choose number of workers: -1 uses all CPUs. If I/O bound, reduce this.
n_jobs = -1
results = Parallel(n_jobs=n_jobs, verbose=5)(delayed(process_file)(p, out_dir) for p in files)

# Summary
oks = [r for r in results if r[1] == 'ok']
errs = [r for r in results if r[1] != 'ok']
print(f"Completed: {len(oks)}; Failed: {len(errs)}")
if errs:
    print('Failures (first 10):')
    for fn, msg in errs[:10]:
        print('-', fn, msg)

Found 112 files to process


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done  96 out of 112 | elapsed:  7.4min remaining:  1.2min


Completed: 112; Failed: 0


[Parallel(n_jobs=-1)]: Done 112 out of 112 | elapsed:  8.3min finished


In [ ]:
N_SPLITS = 5
TEST_RATIO = 0.2
SEED = 42


def split_and_save(data_dir: str,
                   output_dir: str,
                   n_splits: int = N_SPLITS,
                   test_ratio: float = TEST_RATIO,
                   seed: int = SEED):

    files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files in {data_dir}")

    # ── Collect all unique time_ids (only load that column) ──────
    all_tids = set()
    for f in files:
        tids = pd.read_csv(f, usecols=["time_id"])["time_id"].unique()
        all_tids.update(tids)
    all_tids = np.array(sorted(all_tids))
    print(f"{len(all_tids)} unique time_ids, {len(files)} stock files")

    # ── Pre-compute fold assignments using GroupShuffleSplit ──────
    # Feed a dummy array with time_ids as both X and groups
    gss = GroupShuffleSplit(n_splits=n_splits, test_size=test_ratio, random_state=seed)
    fold_splits = []  # list of (train_tid_set, test_tid_set)

    for train_idx, test_idx in gss.split(all_tids, groups=all_tids):
        train_set = set(all_tids[train_idx])
        test_set  = set(all_tids[test_idx])
        assert train_set.isdisjoint(test_set)
        fold_splits.append((train_set, test_set))
        print(f"  Fold {len(fold_splits)-1}: {len(train_set)} train tids, {len(test_set)} test tids")

    # ── Create dirs & writers ────────────────────────────────────
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, "full.parquet")
    full_writer = None

    fold_writers = {"train": [None] * n_splits, "test": [None] * n_splits}
    fold_dirs = []
    for fold in range(n_splits):
        fd = os.path.join(output_dir, f"fold_{fold}")
        os.makedirs(fd, exist_ok=True)
        fold_dirs.append(fd)

    # ── Stream one CSV at a time ─────────────────────────────────
    for i, f in enumerate(files):
        stock_id = os.path.basename(f).replace(".csv", "")
        df = pd.read_csv(f)
        df["stock_id"] = stock_id

        # Write to full.parquet
        table = pa.Table.from_pandas(df, preserve_index=False)
        if full_writer is None:
            full_writer = pq.ParquetWriter(full_path, table.schema)
        full_writer.write_table(table)

        # Write to each fold
        for fold, (train_set, test_set) in enumerate(fold_splits):
            train_chunk = df[df["time_id"].isin(train_set)]
            test_chunk  = df[df["time_id"].isin(test_set)]

            for split, chunk in [("train", train_chunk), ("test", test_chunk)]:
                t = pa.Table.from_pandas(chunk.reset_index(drop=True), preserve_index=False)
                if fold_writers[split][fold] is None:
                    path = os.path.join(fold_dirs[fold], f"{split}.parquet")
                    fold_writers[split][fold] = pq.ParquetWriter(path, t.schema)
                fold_writers[split][fold].write_table(t)

        print(f"[{i+1}/{len(files)}] {stock_id} ({len(df)} rows)")
        del df, table

    # ── Close all writers ────────────────────────────────────────
    if full_writer:
        full_writer.close()
    for split in ("train", "test"):
        for w in fold_writers[split]:
            if w:
                w.close()

    print(f"\nDone ✓ → {output_dir}/")


if __name__ == "__main__":
    split_and_save(
        data_dir   = r"C:\Users\ngdo0466\OneDrive - The University of Sydney (Students)\individual_book_train_denorm",
        output_dir = "processed",
    )